# Create unique ID and concatenate separate years into one dataframe

In [17]:
import os
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.append("../utils")

import config

pd.set_option("display.max_columns", None)


### Import unclipped training geometries

In [18]:
def process_training_geometries(start_year, end_year):
    """
    Process training geometries for a range of years, standardize columns, and combine them into a single GeoDataFrame.
    Preserves the existing inspection_id values and sorts the final result by inspection_id.

    Args:
        start_year (int): The starting year of the range.
        end_year (int): The ending year of the range.

    Returns:
        GeoDataFrame: Combined, processed, and sorted GeoDataFrame.
    """

    def import_training_geometries(year):
        """
        Function to read in buffer geometry data for a given year and convert to Albers CRS.
        """
        training_geometries = os.path.join(
            config.data_dir,
            "training_geometries",
            f"training_geometries_{year}.geojson",
        )
        return gpd.read_file(training_geometries).to_crs(config.albers_crs)

    # Import and process geometries for each year
    inspections_dict = {
        year: import_training_geometries(year)
        for year in range(start_year, end_year + 1)
    }

    # Add a `year` column to each dataframe
    for year, df in inspections_dict.items():
        df["year"] = year

    # Combine all years into one dataframe
    training_inspections = pd.concat(
        inspections_dict.values(), axis=0, ignore_index=True
    )
    
    # Sort by inspection_id to ensure consistent ordering
    training_inspections = training_inspections.sort_values('inspection_id').reset_index(drop=True)

    return training_inspections

In [19]:
training_inspections = process_training_geometries(2019, 2023)
training_inspections

,inspection_id,apn,Date,year,month,status,geometry
0,1,None,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37..."
1,2,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37..."
2,3,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37..."
3,4,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37..."
4,5,None,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37..."
...,...,...,...,...,...,...,...
67575,67576,None,2023-10-31,2023,10,Compliant,"POLYGON ((-35016.13 -349482.83, -34938.314 -34..."
67576,67577,None,2023-11-15,2023,11,Compliant,"POLYGON ((-36511.881 -350331.419, -36432.622 -..."
67577,67578,None,2023-11-15,2023,11,Compliant,"POLYGON ((-38782.099 -346106.741, -38702.857 -..."
67578,67579,None,2023-11-15,2023,11,Compliant,"POLYGON ((-38988.592 -346166.568, -38884.895 -..."


In [20]:
# Write to file
training_inspections.to_file(
    os.path.join(
        config.data_dir,
        "PUZZLE_PIECES",
        "inspections_master_training_geometries.geojson",
    ),
    driver="GeoJSON",
)

### Import clipped geometries

In [21]:
def process_buffer_geometries(start_year, end_year):
    """
    Process buffer geometries for a range of years, standardize columns, and combine them into a single GeoDataFrame.
    Preserves the existing inspection_id values and sorts the final result by inspection_id.

    Args:
        start_year (int): The starting year of the range.
        end_year (int): The ending year of the range.

    Returns:
        GeoDataFrame: Combined, processed, and sorted GeoDataFrame.
    """

    def import_buffer_geometries(year):
        """
        Function to read in buffer geometry data for a given year and convert to Albers CRS.
        """
        buffer_geometries = os.path.join(
            config.data_dir,
            "buffer_geometries",
            f"buffer_geometries_{year}.geojson",
        )
        return gpd.read_file(buffer_geometries).to_crs(config.albers_crs)

    # Import and process geometries for each year
    inspections_dict = {
        year: import_buffer_geometries(year) for year in range(start_year, end_year + 1)
    }

    # Add a `year` column to each dataframe
    for year, df in inspections_dict.items():
        df["year"] = year
        # Reset index to ensure unique indices before concatenation
        inspections_dict[year] = df.reset_index(drop=True)

    # Find common columns across all dataframes
    common_cols = set.intersection(
        *[set(df.columns) for df in inspections_dict.values()]
    )

    # Only keep common columns before concatenation
    for year in inspections_dict:
        inspections_dict[year] = inspections_dict[year][list(common_cols)]

    # Combine all years into one dataframe
    buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)
    
    # Sort by inspection_id to ensure consistent ordering
    buffer_inspections = buffer_inspections.sort_values('inspection_id').reset_index(drop=True)

    return buffer_inspections

In [22]:
buffer_inspections = process_buffer_geometries(2019, 2023)
buffer_inspections

/tmp/ipykernel_560387/4052484543.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)
/tmp/ipykernel_560387/4052484543.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)
/tmp/ipykernel_560387/4052484543.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version

,occupantho,inspecti_3,number_of_dead_trees_within_300_ft_of_residence,project,shift_juli,enginenumber,calfireuni,creationdate,water_source,clearreins,firescopeid,c_removede,community,n_addressd,deckporchelevated,previous_1,appenddate,previousyr,recommend_,e_removedy,calculat_2,apn,prevention_inspectorlastname,keyid,system_upd,address_lo,calculat_3,address_sub_thoroughfare,water_comments,h_cutannua,stationname,l_removelogstumpsembeddedsoil,structuretype,i_reducefu,community_,assigned_to,inspectionstatus,k_removedeaddyingwoodyfuels,partner_di,water_comm,j_allexpos,exteriorsi,editedby,photoid_url,addressvisible,system_created_at,inspectioncount,numberfield2,citationda,textfield2,addressvis,f_removefl,utilitym_1,photoid,address_th,le100number,siteaddres,editstatus,k_removede,month,updated_at,l_removelo,principal_,calculatedglobalid,created_by,address_co,reinspecti,n_displayaddresscontrasting,address_po,deckporchgrade,calculat_1,county,reinspectiondate,stationn_1,coredatafl,nonhabitableoutbuildings,stationnam,year,numberfield1,le100numbe,e_removefl,address_country,water_sour,address__2,b_removeleavesneedlesveg,inspectiondate_calculate,number_of_,comments,prevention_inspectorlastname_other,latitude,exteriorsiding,address_postal_code,structurehabitable,inspection,utilitymis,h_cutannualgrassesforbs,editor,address__1,version,updated_by,b_removele,i_removefuelsusingctguidelines,geometry,calculateduid,c_removedeaddyingtrees,gatecode,system_cre,system_updated_at,firescopei,o_stovepip,address_thoroughfare,battalion,deliverynotificationmethod,textfield1,enginenumb,address_locality,j_exposedw,address_fu,address_admin_area,water_stor,createddat,eaves,ventscreen,waterstora,propanetankdistance,status,fulcrum_id,assigned_t,propertyst,n_displaya,coredata,patiocovercarport,g_relocateexposedwoodpiles,numberofstructures,inspectorp,m_outbuild,createdby,address_suite,a_removebranchesfromstovepipe,calfireunit,creator,o_stovepipemetalscreenopenings,deadtrees3,address_su,core_with_,address_full,inspecti_1,calculatededitor,previousyrinspecteddata,prevention_inspectorfirstname_other,shift,citationnu,inspectorfirstname,photoid_caption,inspection_id,j_exposedwoodpiles,a_removebr,prevention_inspectorfirstname,address_sub_admin_area,shift_julian,roofconstruction,utilitymiscstucturecount,roofconstr,inspectorlastname,escalatetocoordinator,numberfiel,longitude,e_removeflammablegroundcover,creationda,editdate,fenceattachedtostructure,propertystatus,l_logsstum,deckporche,created_at,nonhabitab,i_removefu,recommendc,inspectorposition,m_outbuildingsliquidpropanegas,o_chimneys,fenceattac,calculatededitdate,editeddate,windowpane,can_engine,k_deaddyin,calculat_4,citationnumber,siteaddress,inspecti_2,accessegre,patiocover,yearbuilt,address_ad,responsibi,structuret,numberofst,d_removede,deliveryno,g_relocate,accessegress,previousyrinspectionstatus,f_removeflammablevegetation,report_title,calculated,recommendclearvegetation,report_tit,escalateto,water_storage_size_stored_water_on_individual_parcels_only,occupanthome,inspectionhours,d_removedeaddyinggrassplants,deckporchg,numberfi_1,calculateddate,community_other,structureh,photoid_ca,can_engine_access_water_source,propanetan,Date,globalid,utilitymiscstructuredistance
0,Yes,NaN,NaN,None,None,NaN,SBC,NaT,None,None,None,None,Woodstock,None,None,None,2019-06-05,None,None,None,None,141-010-040,None,NaN,2019-06-05 16:35:26,Santa Ynez,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Wood,None,None,None,NaT,None,None,None,None,Yes - Without Reflective,None,0,None,Brinkerhoff Rd,None,None,None,None,6,2019-06-05 11:57:49,None,None,None,None,US,NaT,None,93460,None,None,SBA,NaT,None,None,None,32,2019,None,0.0,None,None,Private Hydrant- Private Stored Water,Santa Barbara,None,NaT,NaN,None,None,34.669616,None,None,None,2019-06-05,30-50',None,None,None,2.0,None,None,None,"POLYGON ((-2813.179 -371821.557, -2813.179 -37...",None,None,None,2019-05-07 11:52:36,NaT,None,None,None,2,None,None,32,None,None,3462 Brinker

In [23]:
# Sort columns alphabetically
buffer_inspections = buffer_inspections[sorted(buffer_inspections.columns)]

# Reorder columns
front_cols = ["inspection_id", "status", "year", "month"]
end_col   = ["geometry"]
middle_cols = [c for c in buffer_inspections.columns
               if c not in (*front_cols, *end_col)]
buffer_inspections = buffer_inspections[front_cols + middle_cols + end_col]

buffer_inspections

,inspection_id,status,year,month,Date,a_removebr,a_removebranchesfromstovepipe,accessegre,accessegress,address__1,address__2,address_ad,address_admin_area,address_co,address_country,address_fu,address_full,address_lo,address_locality,address_po,address_postal_code,address_su,address_sub_admin_area,address_sub_thoroughfare,address_suite,address_th,address_thoroughfare,addressvis,addressvisible,apn,appenddate,assigned_t,assigned_to,b_removele,b_removeleavesneedlesveg,battalion,c_removede,c_removedeaddyingtrees,calculat_1,calculat_2,calculat_3,calculat_4,calculated,calculateddate,calculatededitdate,calculatededitor,calculatedglobalid,calculateduid,calfireuni,calfireunit,can_engine,can_engine_access_water_source,citationda,citationnu,citationnumber,clearreins,comments,community,community_,community_other,core_with_,coredata,coredatafl,county,created_at,created_by,createdby,createddat,creationda,creationdate,creator,d_removede,d_removedeaddyinggrassplants,deadtrees3,deckporche,deckporchelevated,deckporchg,deckporchgrade,deliveryno,deliverynotificationmethod,e_removedy,e_removefl,e_removeflammablegroundcover,eaves,editdate,editedby,editeddate,editor,editstatus,enginenumb,enginenumber,escalateto,escalatetocoordinator,exteriorsi,exteriorsiding,f_removefl,f_removeflammablevegetation,fenceattac,fenceattachedtostructure,firescopei,firescopeid,fulcrum_id,g_relocate,g_relocateexposedwoodpiles,gatecode,globalid,h_cutannua,h_cutannualgrassesforbs,i_reducefu,i_removefu,i_removefuelsusingctguidelines,inspecti_1,inspecti_2,inspecti_3,inspection,inspectioncount,inspectiondate_calculate,inspectionhours,inspectionstatus,inspectorfirstname,inspectorlastname,inspectorp,inspectorposition,j_allexpos,j_exposedw,j_exposedwoodpiles,k_deaddyin,k_removede,k_removedeaddyingwoodyfuels,keyid,l_logsstum,l_removelo,l_removelogstumpsembeddedsoil,latitude,le100numbe,le100number,longitude,m_outbuild,m_outbuildingsliquidpropanegas,n_addressd,n_displaya,n_displayaddresscontrasting,nonhabitab,nonhabitableoutbuildings,number_of_,number_of_dead_trees_within_300_ft_of_residence,numberfi_1,numberfiel,numberfield1,numberfield2,numberofst,numberofstructures,o_chimneys,o_stovepip,o_stovepipemetalscreenopenings,occupantho,occupanthome,partner_di,patiocover,patiocovercarport,photoid,photoid_ca,photoid_caption,photoid_url,prevention_inspectorfirstname,prevention_inspectorfirstname_other,prevention_inspectorlastname,prevention_inspectorlastname_other,previous_1,previousyr,previousyrinspecteddata,previousyrinspectionstatus,principal_,project,propanetan,propanetankdistance,propertyst,propertystatus,recommend_,recommendc,recommendclearvegetation,reinspecti,reinspectiondate,report_tit,report_title,responsibi,roofconstr,roofconstruction,shift,shift_juli,shift_julian,siteaddres,siteaddress,stationn_1,stationnam,stationname,structureh,structurehabitable,structuret,structuretype,system_cre,system_created_at,system_upd,system_updated_at,textfield1,textfield2,updated_at,updated_by,utilitym_1,utilitymis,utilitymiscstructuredistance,utilitymiscstucturecount,ventscreen,version,water_comm,water_comments,water_sour,water_source,water_stor,water_storage_size_stored_water_on_individual_parcels_only,waterstora,windowpane,yearbuilt,geometry
0,1,Compliant,2019,6,2019-06-05,None,None,yes,None,None,Santa Barbara,CA,None,US,None,3462 Brinkerhoff Rd Santa Ynez Santa Barbara C...,None,Santa Ynez,None,93460,None,3462,None,None,None,Brinkerhoff Rd,None,Yes - Without Reflective,None,141-010-040,2019-06-05,None,None,None,None,2,None,None,None,None,None,None,None,None,None,None,None,None,SBC,None,yes,None,None,None,None,None,None,Woodstock,None,None,None,None,None,SBA,2018-05-25 16:25:06,None,None,None,NaT,NaT,None,None,None,0,No Deck/Porch,None,No Deck/Porch,None,Hardcopy,None,None,None,None,Unenclosed,NaT,None,None,None,None,32,NaN,None,None,Wood,None,None,None,No Fence,None,None,None,d9a4d031-d16d-4c84-b32c-f3fcd7d945df,None,None,None,None,None,None,None,None,None,30 min,0.0,NaN,2019-06-05,None,NaT,None,

In [24]:
# Write to file
buffer_inspections.to_file(
    os.path.join(config.data_dir, "PUZZLE_PIECES", "inspections_master.geojson"),
    driver="GeoJSON",
)

## Check that the inpection ids match for training geometry and buffer geometries

In [25]:
# Create dataframe of just compliance status and inspection_id
status_training = training_inspections.reindex(columns=["inspection_id", "status"])
status_training["status"] = status_training["status"].map(
    {"Compliant": 0, "Non-Compliant": 1}
)

In [26]:
status_buffer = buffer_inspections.reindex(columns=["inspection_id", "status"])
status_buffer["status"] = status_buffer["status"].map(
    {"Compliant": 0, "Non-Compliant": 1}
)

In [27]:
status_training_sorted = status_training.sort_values('inspection_id').reset_index(drop=True)
status_buffer_sorted = status_buffer.sort_values('inspection_id').reset_index(drop=True)
are_equal = status_training_sorted.equals(status_buffer_sorted)
are_equal

True